# Pancancer enrichment analysis step 2: Find enriched pathways

In [1]:
import cptac
import cptac.utils as ut
import pandas as pd
import numpy as np
import gseapy as gp
import os
import datetime

In [2]:
TIME_START = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

In [3]:
# Read in the file from step 1
data = pd.read_csv("step1/tumor_changes.tsv", sep='\t', index_col=0)

# Only keep genes with P values below the cutoff
data = data[data["reject_null"]]

In [4]:
# For each cancer, find enriched pathways for genes that changed (either up or down)   
all_enrichments = pd.DataFrame()

for cancer_type in data["cancer_type"].unique():
    cancer_subset = data[data["cancer_type"] == cancer_type]
    changed_proteins = cancer_subset["protein_str"].tolist()
    enriched_pathways = gp.enrichr(
        gene_list=changed_proteins,
        description="changed_in_tumor",
        gene_sets="Reactome_2016",
        outdir=None)
    cancer_enriched = enriched_pathways.res2d.assign(cancer_type=cancer_type)
    all_enrichments = all_enrichments.append(cancer_enriched)

In [5]:
# Make a column of just pathway IDs
all_enrichments = all_enrichments.assign(pathway_id=all_enrichments["Term"].str.rsplit(" ", n=1, expand=True)[1])

# Make a column of overlap proportions
overlaps = all_enrichments["Overlap"].str.split("/", expand=True).astype("int")
overlap_props = overlaps[0] / overlaps[1]
all_enrichments = all_enrichments.assign(overlap_prop=overlap_props)

In [6]:
# Make a table with a list of all pathways, and:
# - The number of cancer types for which that cancer type is enriched
# - The average of the adjusted p values for that pathway
# - The average overlap proportion for that pathway
enrichment_summary = all_enrichments[["pathway_id", "Term"]].drop_duplicates(keep="first")

times_enriched = enrichment_summary["pathway_id"].apply(
    lambda x: all_enrichments[all_enrichments["pathway_id"] == x].shape[0])

avg_p_vals = enrichment_summary["pathway_id"].apply(
    lambda x: all_enrichments.loc[all_enrichments["pathway_id"] == x, "Adjusted P-value"].mean())

avg_overlap_props = enrichment_summary["pathway_id"].apply(
    lambda x: all_enrichments.loc[all_enrichments["pathway_id"] == x, "overlap_prop"].mean())

enrichment_summary = enrichment_summary.assign(
    times_enriched=times_enriched,
    avg_adj_p=avg_p_vals,
    avg_overlap_prop=avg_overlap_props)

enrichment_summary = enrichment_summary.sort_values(by=["times_enriched", "avg_adj_p"], ascending=[False, True])
enrichment_summary = enrichment_summary.reset_index(drop=True)

In [7]:
# In the omics data, make all increases +1 and decreases -1
# Because these are ratioed abundances, we can't compare magnitudes between
# genes--we can only compare positive or negative
data = data.assign(simplified_change=data["change"])

data.loc[data["change"] > 0, "simplified_change"] = 1
data.loc[data["change"] < 0, "simplified_change"] = -1
data.loc[data["change"] == 0, "simplified_change"] = 0

In [8]:
# Visualize selected pathways (this will show which genes are up or down)
to_viz = enrichment_summary["pathway_id"][0:1]

cancer_types_list = []
pathways_list = []
urls_list = []

for pathway in to_viz:
    for cancer_type in data["cancer_type"].unique():
        omics = data.loc[data["cancer_type"] == cancer_type, "simplified_change"]
        omics.name = f"{cancer_type}_change_tumor_normal"
        url = ut.reactome_pathway_overlay(omics, pathway, open_browser=False)
        cancer_types_list.append(cancer_type)
        pathways_list.append(pathway)
        urls_list.append(url)
        

urls_df = pd.DataFrame({
    "cancer_type": cancer_types_list,
    "pathway": pathways_list,
    "url": urls_list})

In [9]:
# Save the results
if not os.path.isdir("step2"):
    os.mkdir("step2")
    
urls_df.to_csv(f"step2/urls_{TIME_START}.tsv", sep='\t')